<a href="https://colab.research.google.com/github/Ddkaba/IAD_Curs/blob/main/IAD_Cursach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.2. Переразметить исходный набор ТМИ МКА на случай бинарной
классификации: штатное функционирование, нештатное ф-е (объединить вектора с
метками отказ и сбой). Разработать приложение на языке Python с использованием
необходимых библиотек машинного и глубокого обучения на основе бинарной
классификационной модели, заданной в варианте x, которое определяет техническое
состояние малого космического аппарата: штатное ф-е 0, нештатное ф-е 1, на основе данных его ТМИ.

6.2. (для вариантов 1.2) Построить автокодировщик, определяющий аномалии – вектора, соответствующие нештатным состояниям МКА, на основании базового
нейросетевого классификатора, заданного в варианте. Обучить с валидацией на
размеченного наборе данных ТМИ, не используя эталонные метки. Протестировать
на тестовой подвыборке размеченного набора ТМИ и получить классификационные
метрики с учетом значений имеющихся эталонных метрик: сравнить результаты –
метки штатных и нештатных состояний, полученные обученным обученным
автокодировщиком, определяющим аномалии, и значения эталонных меток для
тестового поднабора размеченной выборки.  
Гибридная нейросетевая модель: последовательностное соединение  
одномерных сверточных и рекуррентных GRU нейросетевых слоев с
полносвязным классификатором: самостоятельная реализация.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
dataset = pd.read_csv(
    "https://raw.githubusercontent.com/Ddkaba/IAD_Curs/main/dataset.csv")
dataset.columns = [c.split(",")[0] if "," in str(c) else c for c in dataset.columns]
if "No" in dataset.columns:
    dataset = dataset.drop(columns=["No"])
# Переразметка на бинарную классификацию: штатное 0, нештатное 1 (отказ и сбой объединены)
target_col = dataset.columns[-1]
dataset[target_col] = (dataset[target_col] > 0).astype(int)

In [ ]:
TARGET = dataset.columns[-1]

print("Общая информация")
print(dataset.info())

print(f"Количество записей (объектов): {dataset.shape[0]}")
print(f"Количество признаков (фич): {dataset.shape[1]}")

print("\nНазвания столбцов:")
print(dataset.columns.tolist())

print("\nТипы данных:")
print(dataset.dtypes)

print("\nПропущенные значения:")
missing_values = dataset.isnull().sum()
print(missing_values)
print(f"Общее количество пропущенных значений: {missing_values.sum()}")

print("Целевая переменная")
if TARGET in dataset.columns:
    print(f"\nЦелевая переменная: {TARGET}")
    print(f"Тип данных целевой переменной: {dataset[TARGET].dtype}")
    unique_values = dataset[TARGET].unique()
    print(f"Уникальные значения целевой переменной (первые 20): {unique_values[:20]}")
    print(f"Всего уникальных значений: {unique_values.size}")
    if dataset[TARGET].nunique() <= 20:
        print("Распределение классов:")
        print(dataset[TARGET].value_counts())
        print("Процентное соотношение классов:")
        print(dataset[TARGET].value_counts(normalize=True) * 100)

print("Статистика")
print(dataset.describe())

In [ ]:
feat = [c for c in dataset.columns if c != TARGET]
n = len(feat)
k = 4
size = (n + k - 1) // k
chunks = [feat[i : i + size] for i in range(0, n, size)]

for i, cols in enumerate(chunks):
    block = cols + [TARGET]
    corr = dataset[block].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
    plt.title("Корреляционная матрица (признаки {}-{}, + {})".format(i * size + 1, min((i + 1) * size, n), TARGET))
    plt.tight_layout()
    plt.show()

In [ ]:
corr_with_target = dataset[feat + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET, errors="ignore")
top2 = corr_with_target.abs().nlargest(2).index.tolist()
x_col, y_col = top2[0], top2[1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=dataset, x=x_col, y=y_col, hue=TARGET, alpha=0.7)
plt.title("Диаграмма рассеивания: {} vs {} (цвет = {})".format(x_col, y_col, TARGET))
plt.tight_layout()
plt.show()

In [ ]:
feature_columns = dataset.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in feature_columns:
    feature_columns.remove(TARGET)

_ = dataset[feature_columns].hist(
    bins=10,
    figsize=(20, 15),
    grid=False,
    edgecolor="black",
)
plt.suptitle("Распределение числовых признаков", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def standardize_features(dataset, target):
    """Стандартизация числовых признаков (z = (x - mean) / std), без целевой."""
    if target not in dataset.columns:
        raise ValueError(f"Целевая переменная '{target}' отсутствует в датасете.")
    base = dataset.copy()
    feature_cols = base.select_dtypes(include=[np.number]).columns.tolist()
    if target in feature_cols:
        feature_cols.remove(target)
    if not feature_cols:
        raise ValueError("Нет числовых признаков для стандартизации.")
    scaler = StandardScaler()
    scaled = scaler.fit_transform(base[feature_cols])
    standardized_df = pd.DataFrame(scaled, columns=feature_cols, index=base.index)
    out = pd.concat([standardized_df, base[[target]]], axis=1)
    print("Стандартизированы признаки:", len(feature_cols))
    print("Средние ≈ 0:", standardized_df.mean().round(4).tolist()[:5], "...")
    print("Ст. откл. ≈ 1:", standardized_df.std(ddof=0).round(4).tolist()[:5], "...")
    return out

dataset_preprocessed = standardize_features(dataset, TARGET)
feat_cols = [c for c in dataset_preprocessed.columns if c != TARGET]
_ = dataset_preprocessed[feat_cols].hist(bins=10, figsize=(20, 15), grid=False, edgecolor="black")
plt.suptitle("Распределение числовых признаков (после стандартизации)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
source_df = dataset
X_num = source_df.drop(columns=[TARGET]).select_dtypes(include=[np.number])
y = source_df[TARGET]

mi_scores = mutual_info_classif(X_num, y, random_state=0)
scores_df = (
    pd.DataFrame({"feature": X_num.columns, "score": mi_scores})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print("Оценки информативности (mutual information), по убыванию:")
print(scores_df)

plt.figure(figsize=(10, 12))
sns.barplot(data=scores_df, x="score", y="feature", color="#1f77b4")
plt.title("Mutual information: scores")
plt.tight_layout()
plt.show()

K = 5
selected_features = scores_df["feature"].head(K).tolist()
print(f"Топ-{K} признаков:")
print(selected_features)

In [ ]:
# П.3: Исходный и преобразованный наборы для анализа и классификации
# Исходный — все признаки; преобразованный — отбор по MI для лучшей точности

N_TOP = 30
top_features = scores_df["feature"].head(N_TOP).tolist()

dataset_orig = dataset.copy()
dataset_transformed = dataset[top_features + [TARGET]].copy()

print("Исходный набор: {} признаков".format(dataset_orig.shape[1] - 1))
print("Преобразованный набор (топ-{} по MI): {}".format(N_TOP, list(dataset_transformed.columns)))

# Стандартизация только для преобразованного набора (отбор признаков + стандартизация)
dataset_transformed_preprocessed = standardize_features(dataset_transformed, TARGET)

In [ ]:
# П.4: Проверка сбалансированности классов
counts = dataset[TARGET].value_counts()
props = dataset[TARGET].value_counts(normalize=True)
print("Распределение классов:")
print(counts)
print("\nДоли:")
print(props)
imbalance_ratio = props.max() / props.min()
if imbalance_ratio > 2.0:
    print("\nНабор несбалансирован (отношение макс/мин > 2). Оцениваем balanced_accuracy и ROC-AUC.")
else:
    print("\nНабор умеренно сбалансирован.")

In [ ]:
def fit_eval(df, name, scale=False):
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
    else:
        X_train, X_test = X_train.values, X_test.values
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    print("---", name, "---")
    print(classification_report(y_test, y_pred, target_names=["штатное", "нештатное"]))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))

fit_eval(dataset_orig, "Исходный набор", scale=True)
print()
fit_eval(dataset_transformed_preprocessed, "Преобразованный набор (топ по MI + стандартизация)", scale=False)